<a href="https://colab.research.google.com/github/alexaK88/Fun_tasks_for_the_weekend/blob/main/word2vec_numpy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import re
import numpy as np
from tqdm import tqdm
from collections import Counter

In [15]:
text_path = "/content/drive/MyDrive/star_rover.txt"

Here we will focus on one of my favourite books - The Jacket (aka, Star-Rover) by Jack London.


Pipeline:
1. Load and tokenize the text
2. Build vocabulary
3. Encode corpus
4. Generate training pairs
5. Implement negative sampler
6. Initialize embeddings
7. Implement sigmoid
8. Implement forward pass
9. Implement loss
10. Implement gradients
11. Implement SGD update
12. Train

### Data Pipeline

- Load text
- Preprocess and tokenize
- Build vocabulary
- Encode corpus
- Generate training pairs

In [16]:
def load_text(path):
  with open(path, "r", encoding="utf-8") as f:
    text = f.read().lower() # lowercase the whole text
  return text

def tokenize(text):
    tokens = re.findall(r"\b[a-z]+\b", text) # we will only keep words, no punctuation or anything else
    return tokens

In [17]:
def build_vocab(tokens, min_count=5):
    counts = Counter(tokens)
    vocabulary = [w for w, c in counts.items() if c >= min_count]

    word2ix = {w:i for i, w in enumerate(vocabulary)}
    idx2word = {i:w for w, i in word2ix.items()}

    word_freq = np.array([counts[w] for w in vocabulary])

    return word2ix, idx2word, word_freq

In [18]:
# creating list of tokens
text = load_text(text_path)
tokens = tokenize(text)

# building vocabulary - all words/tokens are unique
word2idx, idx2word, word_freq = build_vocab(tokens)
# generating training pairs; we are creating pairs like (center_word, context_word)
encoded = np.array([word2idx[w] for w in tokens if w in word2idx],
                   dtype=np.int32)

Negative sampling steps:
1. Compute word frequencies
2. Apply Mikolov distribution
3. Randomly sample K negative words

In [19]:
class NegativeSampler:
    def __init__(self, word_freq, power=0.75):
        freq = np.array(word_freq) ** power
        self.prob = freq / freq.sum()

    def sample(self, K):
        return np.random.choice(len(self.prob), size=K, p=self.prob)

def subsample_tokens(encoded, word_freq, t=1e-5):
    total = np.sum(word_freq)
    freq = np.array(word_freq) / total

    keep_probs = np.sqrt(t / freq) + (t / freq)

    keep_probs = np.minimum(1.0, keep_probs)

    mask = np.random.rand(len(encoded)) < keep_probs[encoded]

    return encoded[mask]

In [20]:
def generate_skipgram_pairs(encoded, max_window=5):
    for i in range(len(encoded)):
        center = encoded[i]

        window = np.random.randint(1, max_window + 1)

        start = max(0, i - window)
        end = min(len(encoded), i + window + 1)

        for j in range(start, end):
            if i != j:
                yield center, encoded[j]

In [21]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))


class Word2VecSGNS:

    def __init__(self, vocab_size, embed_dim=100, lr=0.025):

        self.V = vocab_size
        self.D = embed_dim
        self.lr = lr

        # input embeddings
        self.W_in = np.random.randn(self.V, self.D) * 0.01

        # output embeddings
        self.W_out = np.random.randn(self.V, self.D) * 0.01

    def train_pair(self, center, context, negatives):
        """
        One SGD update step.
        """

        v_c = self.W_in[center]
        v_o = self.W_out[context]

        neg_vectors = self.W_out[negatives]

        # ---------- Forward ----------
        pos_score = np.dot(v_o, v_c)
        neg_scores = neg_vectors @ v_c

        pos_sig = sigmoid(pos_score)
        neg_sig = sigmoid(neg_scores)

        # Loss (optional for logging)
        loss = -np.log(pos_sig + 1e-10) \
               -np.sum(np.log(1 - neg_sig + 1e-10))

        # ---------- Gradients ----------

        # positive gradient
        grad_pos = pos_sig - 1

        # negative gradients
        grad_neg = neg_sig

        # gradient wrt center embedding
        grad_center = (
            grad_pos * v_o +
            np.sum(grad_neg[:, None] * neg_vectors, axis=0)
        )

        # ---------- Updates ----------

        # update output embeddings
        self.W_out[context] -= self.lr * grad_pos * v_c

        self.W_out[negatives] -= (
            self.lr * grad_neg[:, None] * v_c
        )

        # update center embedding
        self.W_in[center] -= self.lr * grad_center

        return loss


In [23]:
WINDOW = 5
NEG_SAMPLES = 5
EPOCHS = 10

encoded = subsample_tokens(encoded, word_freq)
sampler = NegativeSampler(word_freq)

model = Word2VecSGNS(len(word2idx), embed_dim=100)

for epoch in range(EPOCHS):

    total_loss = 0

    for center, context in tqdm(
        generate_skipgram_pairs(encoded, WINDOW)
    ):

        negatives = sampler.sample(NEG_SAMPLES)

        loss = model.train_pair(center, context, negatives)
        total_loss += loss

    print(f"Epoch {epoch+1} loss: {total_loss:.2f}")


23024it [00:03, 6052.86it/s]


Epoch 1 loss: 95739.11


22842it [00:02, 7693.36it/s]


Epoch 2 loss: 93751.81


22992it [00:02, 7755.05it/s]


...

Epoch 8 loss: 47749.68


22971it [00:04, 5271.02it/s]


Epoch 9 loss: 46225.06


23263it [00:02, 7778.88it/s]

Epoch 10 loss: 46434.12
